# 🏥 TRICARE RAG Q&A 통합 파이프라인 v4 (integrated5)

## 개요
- `tricare_rag_integrated4.ipynb` 기반으로 전처리·검색 방식 개선한 버전임
- 신규 PDF 3종은 다음 창에서 추가 예정: `RAG_PDF_DOCS` 리스트와 `TABLE_PDF_DOCS`에 경로만 추가하면 됨

## 변경 사항 (integrated4 대비)

| Section | 변경 내용 |
|---|---|
| **Section 4** | `clean_text()` · `normalize_cell()` · `enrich_tricare_text()` · `is_noise_line()` 추가 |
| **Section 5** | RecursiveCharacterTextSplitter → **단락(`\\n\\n`) 단위 직접 분리** (80자 미만 제거) |
| **Section 8** | 표 셀에 `normalize_cell()` 적용, 빈 행 제거 |
| **Section 10** | `similarity` → **MMR** 검색, `normalize_question()` 추가 |
| **Section 11** | Term Locking 프롬프트 추가, `make_rag_chain_v2()` 추가 (MMR 완전 통합) |

## 나머지 3개 PDF 추가 방법
```python
# Section 2: RAG_PDF_DOCS에 경로 추가
RAG_PDF_DOCS = [
    ...기존 목록...,
    os.path.join(DATA_DIR, '추가할파일.pdf'),  # ← 여기 추가
]

# Section 4: LOCATION_MAP에 추가 (OCONUS/CONUS인 경우만)
LOCATION_MAP = {
    ...기존 매핑...,
    '추가할파일.pdf': 'OCONUS',  # ← BOTH면 생략 가능
}
```

## 팀별 벡터 DB 경로 (통합 시 참고)
| 팀원 | persist_directory | collection_name |
|------|------------------|-----------------|
| TRICARE (원아) | `./chroma_db`, `./chroma_db2` | `tricare_rag`, `tricare_cost_tables` |
| Cigna (은우님) | `./chroma_cigna_latest`, `./chroma_cigna_all` | `cigna_latest`, `cigna_all` |
| Allianz (수진님) | `./vectordb` | `allianz_care` |

## 전체 실행 순서
1. 패키지 설치 → 환경 설정 → 파일 경로 설정
2. 임베딩 모델 초기화
3. **전처리 유틸 함수 정의** (clean_text, normalize_cell, enrich_tricare_text, is_noise_line) → PDF 로드
4. **단락 단위 청킹** (RecursiveCharacterTextSplitter는 CSV에만 사용)
5. CSV 로드 → Document 변환 → 청킹
6. 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)
7. **normalize_cell 적용** PDF 표 추출 + Health_Plan_Costs.csv → 표 전용 벡터 저장소 생성 (`chroma_db2`)
8. **MMR Retriever** + normalize_question 설정
9. LLM + RAG 체인 구성 (Term Locking 프롬프트 추가)
10. Demo 실행


---
## Section 0 · 패키지 설치

### 개요
- 처음 실행 시에만 필요함
- 주석 해제 후 실행하면 됨


In [5]:
# !pip install langchain langchain-community langchain-chroma langchain-openai \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q


---
## Section 1 · 환경 설정

### 개요
- `.env` 파일에서 `OPENAI_API_KEY`를 읽어옴
- 벡터 DB 경로·컬렉션명을 상수로 분리해 관리함
  - `PERSIST_TEXT` / `COLLECTION_TEXT`: 텍스트+CSV 통합 저장소
  - `PERSIST_TABLE` / `COLLECTION_TABLE`: 비용 표 전용 저장소
- 팀 통합 벡터 DB 병합 시 이 상수들을 기준으로 경로를 맞춤


In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

# 벡터 DB 경로·컬렉션명 상수
# 팀 통합 시 이 두 상수만 보면 어느 팀원 DB인지 바로 알 수 있도록 tricare_ 프리픽스 유지
PERSIST_TEXT = './chroma_db'
PERSIST_TABLE= './chroma_db2'   # 표 내용도 들어감
COLLECTION_TEXT  = 'tricare_rag'
COLLECTION_TABLE = 'tricare_cost_tables'


✅ API Key 로드: 성공


---
## Section 2 · 파일 경로 설정

### 개요
- [변경] 기존 `PDF_COSTS`, `PDF_HANDBOOK` 개별 변수 방식 → `RAG_PDF_DOCS` 리스트 방식으로 교체함
  - 신규 PDF가 추가될 때 리스트에 경로 한 줄만 추가하면 되는 구조임
  - 유지보수성이 크게 향상됨
- `RAG_PDF_DOCS`: `chroma_db`(텍스트 저장소)에 들어갈 PDF 목록
- `TABLE_PDF_DOCS`: `chroma_db2`(표 전용 저장소)에 추가로 표 추출할 PDF 목록
  - `Costs_Fees.pdf`는 두 리스트 모두에 포함됨 (텍스트 + 표 이중 수록 의도적)
  - `TRICARE_ADDP_HB.pdf`는 비용 표가 없으므로 `TABLE_PDF_DOCS`에서 제외함
- 경로는 본인 로컬 환경에 맞게 수정할 것

### 신규 추가 파일 (integrated2 대비)
| 파일 | 저장소 | 이유 |
|---|---|---|
| `Overseas_HB.pdf` | RAG만 | 해외 절차·한국 Near Patient Program 등 서술형 |
| `Pharmacy_HB.pdf` | RAG + TABLE | 본문 서술 + 비용표 모두 존재 |
| `NGR_HB.pdf` | RAG + TABLE | 본문 서술 + 활성화 상태별 플랜 옵션 표 |
| `TRICARE_ADDP_HB.pdf` | RAG만 | ADSM 비용 없는 프로그램, 비용 표 없음 |
| `ADDP_Brochure.pdf` | RAG만 | 개요 서술 |
| `Maternity_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `Retiring_AD_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `Retiring_NGR_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `QLEs_FS.pdf` | RAG만 | 서술형 팩트시트 |

In [ ]:
# 경로 앞부분을 변수로 분리해 수정이 필요할 때 한 곳만 바꾸면 됨
DATA_DIR = r'C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data'
#C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data
#ex. C:\SKN-_24\제 3차 프로젝트\Care-Insure-Bot\data\ADDP_Brochure_FINAL_122624_508c.pdf
# chroma_db (tricare_rag) 에 텍스트 청킹으로 들어갈 PDF 목록
# 신규 PDF 추가 시 이 리스트에 경로 한 줄만 추가하면 됨
RAG_PDF_DOCS = [
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),
    os.path.join(DATA_DIR, 'TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf'),
    os.path.join(DATA_DIR, 'Overseas_HB(해외 프로그램 안내서).pdf'),                  # 신규 추가
    os.path.join(DATA_DIR, 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'),           # 신규 추가
    os.path.join(DATA_DIR, 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'), # 신규 추가
    os.path.join(DATA_DIR, 'TFL_HB(평생 트라이케어).pdf'),                            # 신규 추가
    os.path.join(DATA_DIR, 'TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf'), # 신규 추가
    os.path.join(DATA_DIR, 'ADDP_Brochure_FINAL_122624_508c.pdf'),                   # 신규 추가
    os.path.join(DATA_DIR, 'Maternity_Br (1).pdf'),                                  # 신규 추가
    os.path.join(DATA_DIR, 'Retiring_AD_Br.pdf'),                                    # 신규 추가
    os.path.join(DATA_DIR, 'Retiring_NGR_Br.pdf'),                                   # 신규 추가
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),                                       # 신규 추가
    os.path.join(DATA_DIR, 'QLEs_FS (2).pdf'),                                       # 신규 추가
]

# chroma_db2 (tricare_cost_tables) 에 pdfplumber 표 추출을 적용할 PDF 목록
# 비용 표가 실제로 존재하는 PDF만 포함함
# TRICARE_ADDP_HB는 ADSM 비용 없는 프로그램이라 비용 표가 없으므로 제외함
TABLE_PDF_DOCS = [
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),
    os.path.join(DATA_DIR, 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'),           # 신규 추가
    os.path.join(DATA_DIR, 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'), # 신규 추가
    os.path.join(DATA_DIR, 'TFL_HB(평생 트라이케어).pdf'),                            # 신규 추가
    os.path.join(DATA_DIR, 'Plans_Overview_FS (1).pdf'),                             # 신규 추가 (전체가 표 구조)
]

# CSV 파일 경로 (integrated2와 동일)
CSV_MENTAL = os.path.join(DATA_DIR, 'mental_health_services.csv')
CSV_COSTS  = os.path.join(DATA_DIR, 'Health_Plan_Costs.csv')
CSV_PLANS  = os.path.join(DATA_DIR, 'TricarePlans.csv')
CSV_EXCLUSIONS = os.path.join(DATA_DIR, 'tricare_exclusions.csv')

# 파일 존재 여부 확인
print('[ RAG PDF 목록 ]')
for path in RAG_PDF_DOCS:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {os.path.basename(path)}')

print('\n[ TABLE PDF 목록 ]')
for path in TABLE_PDF_DOCS:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {os.path.basename(path)}')

print('\n[ CSV 목록 ]')
for path in [CSV_MENTAL, CSV_COSTS, CSV_PLANS, CSV_EXCLUSIONS]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {os.path.basename(path)}')


[ RAG PDF 목록 ]
  ❌ Costs_Fees.pdf
  ❌ TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf
  ❌ Overseas_HB(해외 프로그램 안내서).pdf
  ❌ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  ❌ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
  ❌ TFL_HB(평생 트라이케어).pdf
  ❌ TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf
  ❌ ADDP_Brochure_FINAL_122624_508c.pdf
  ❌ Maternity_Br (1).pdf
  ❌ Retiring_AD_Br.pdf
  ❌ Retiring_NGR_Br.pdf
  ❌ Costs_Fees.pdf
  ❌ QLEs_FS (2).pdf

[ TABLE PDF 목록 ]
  ❌ Costs_Fees.pdf
  ❌ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  ❌ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
  ❌ TFL_HB(평생 트라이케어).pdf
  ❌ Plans_Overview_FS (1).pdf

[ CSV 목록 ]
  ❌ mental_health_services.csv
  ❌ Health_Plan_Costs.csv
  ❌ TricarePlans.csv
  ❌ tricare_exclusions.csv


---
## Section 3 · 임베딩 모델 초기화 — `BAAI/bge-m3`

### 개요
- 100개 이상의 언어를 동일한 벡터 공간에 매핑하는 다국어 임베딩 모델임
- 한국어 질문과 영어 문서 간 유사도 매칭이 가능함
- 팀원 Cigna 코드와 동일한 모델을 사용해 임베딩 방식을 통일함
- GPU 있으면 `device: cuda`로 변경하면 속도가 빨라짐


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}  # GPU 있으면 'cuda'로 변경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')


KeyboardInterrupt: 

---
## Section 4 · PDF 로드 — PyMuPDFLoader

### 개요
- [변경] 기존 2개 개별 로더 → `RAG_PDF_DOCS` 리스트 순회 방식으로 교체함
  - PDF가 추가되어도 Section 2의 리스트만 수정하면 이 셀은 그대로 사용 가능함
- `metadata.source_file`을 각 청크에 추가해 어느 파일에서 왔는지 항상 추적 가능하게 함
- `PyMuPDFLoader`는 페이지 단위 추출이며 `metadata.page`가 자동 포함됨

⬇️⬇️⬇️ 내용 수정함

## Section 4 · PDF 로드 — PyMuPDFLoader (수정 후)

### 개요
- [변경] 기존 2개 개별 로더 → `RAG_PDF_DOCS` 리스트 순회 방식으로 교체함
  - PDF가 추가되어도 Section 2의 리스트만 수정하면 이 셀은 그대로 사용 가능함
- `metadata.source_file`: 어느 파일에서 왔는지 팀 통합 시 출처 추적용
- `metadata.location`: OCONUS(해외) 적용 여부 태그
  - `OCONUS`: 해외 전용 문서 (Overseas_HB, TOP_Handbook)
  - `CONUS`: 미국 내 기준 문서 (Choices_US_HB)
  - `BOTH`: 미국·해외 공통이나 일부 규칙 다름 (TFL, Medicare, NGR, ADDP 등)
  - 매핑에 없는 파일은 기본값 `BOTH`
- `PyMuPDFLoader`는 페이지 단위 추출이며 `metadata.page`가 자동 포함됨


In [ ]:
#from langchain_community.document_loaders import PyMuPDFLoader
#
#all_pdf_docs = []
#
## RAG_PDF_DOCS 리스트를 순회하며 일괄 로드
## 신규 PDF가 추가되어도 Section 2의 리스트만 수정하면 이 셀은 수정 불필요
#for pdf_path in RAG_PDF_DOCS:
#    docs = PyMuPDFLoader(pdf_path).load()
#    # 파일명을 source_file 메타데이터로 추가해 팀 통합 시 출처 구분 용이하게 함
#    for doc in docs:
#        doc.metadata['source_file'] = os.path.basename(pdf_path)
#    all_pdf_docs.extend(docs)
#    print(f'  ✅ {os.path.basename(pdf_path)}: {len(docs)}페이지')
#
#print(f'\n📄 총 PDF 페이지 합계: {len(all_pdf_docs)}페이지')
#print(f'\n--- 샘플 메타데이터 ---')
#print(all_pdf_docs[0].metadata)


# 내용 수정: 일부 pdf - 보험 혜택 가이드라인 중 주한미군&한국 내 적용이 일부만 되는 것이 있기 때문

In [ ]:
import re
from langchain_community.document_loaders import PyMuPDFLoader

# ── 전처리 유틸 함수 (수진님 ingest.py 참고해서 TRICARE 버전으로 작성) ──

def clean_text(text: str) -> str:
    """불필요한 유니코드·공백·줄바꿈 정제"""
    text = text.replace('\xa0', ' ')   # non-breaking space
    text = text.replace('\u200b', ' ') # zero-width space
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def normalize_cell(cell: str) -> str:
    """표 셀 ✓/✗/빈칸 처리 — 보험 표에서 체크마크를 LLM이 해석하기 좋은 텍스트로 정규화"""
    if not cell:
        return 'N/A'
    cell = cell.strip()
    if cell in ['✓', '√', 'v', 'V', '●', 'Yes', 'yes', 'Y']:
        return 'Covered'
    if cell in ['✗', '×', 'x', 'X', '✘', 'No', 'no', 'N']:
        return 'Not covered'
    return cell


# TRICARE 도메인 검색 향상용 태그
# 이 태그를 본문 뒤에 붙이면 한국어 질문 → 영어 문서 벡터 간 유사도가 높아짐
TRICARE_SEARCH_TAGS = [
    'coverage', 'covered', 'not covered', 'benefit', 'cost', 'copay',
    'deductible', 'enrollment fee', 'cost-share', 'prior authorization',
    'TRICARE Prime', 'TRICARE Select', 'TRICARE For Life',
    'Group A', 'Group B', 'active duty', 'retiree', 'reserve',
    'OCONUS', 'overseas', 'USFK', '주한미군',
    '보장', '비용', '본인부담금', '사전승인', '해외',
]


def enrich_tricare_text(text: str, fname: str) -> str:
    """본문 뒤에 search_tags 블록 추가 — bge-m3 임베딩에서 한/영 매칭률 향상용"""
    location = LOCATION_MAP.get(fname, 'BOTH')
    tags = '\n'.join([
        '[search_tags]',
        'insurer: TRICARE',
        f'location: {location}',
        'keywords: ' + ', '.join(TRICARE_SEARCH_TAGS),
    ])
    return f'{text}\n\n{tags}'


def is_noise_line(line: str) -> bool:
    """URL 반복·컬럼 헤더 반복 등 노이즈 행 여부 판단"""
    noise_patterns = [
        r'^group a.*group b',
        r'^covered service.*group',
        r'^are you in group',
        r'^this is an overview',
        r'^visit www\.tricare',
        r'^for more information.*go to',
        r'^updated (january|february|march|april|may|june|july|august|september|october|november|december)',
    ]
    return any(re.search(p, line.lower()) for p in noise_patterns)


# ── LOCATION_MAP ──
# 파일명 → location 태그 매핑
# 나머지 3개 PDF 추가 시 이 딕셔너리에 한 줄만 추가하면 됨
# TODO: 다음 창에서 추가할 파일들 (아래 주석 해제 후 사용)
# 'TFL_HB(평생 트라이케어).pdf':                         'BOTH',  ← 이미 기본값 BOTH라 생략 가능
# 'Medicare_Turning_65_Br.pdf':                          'BOTH',  ← 이미 기본값 BOTH라 생략 가능
# 'Medicare_Under_65_Br (7).pdf':                        'BOTH',  ← 이미 기본값 BOTH라 생략 가능
# 'Choices_US_HB(미국 내 tricare 선택 가이드).pdf':       'CONUS', ← CONUS만 명시 필요
# 'Plans_Overview_FS (1).pdf':                           'BOTH',  ← TABLE_PDF_DOCS에만 있음
LOCATION_MAP = {
    'Overseas_HB(해외 프로그램 안내서).pdf':                                'OCONUS',
    'TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf':                      'OCONUS',
    'Choices_US_HB(미국 내 tricare 선택 가이드).pdf':                       'CONUS',
    # 나머지 PDF 3종: 기본값 'BOTH'와 동일하므로 명시 생략 가능
    # 다음 창에서 추가될 파일 중 OCONUS/CONUS가 있다면 여기 추가할 것
}

all_pdf_docs = []

# 루프 하나에서 source_file + location + search_tags 동시 처리
# clean_text + is_noise_line으로 전처리 후 enrich_tricare_text로 search_tags 추가
for pdf_path in RAG_PDF_DOCS:
    docs = PyMuPDFLoader(pdf_path).load()
    fname = os.path.basename(pdf_path)
    loc   = LOCATION_MAP.get(fname, 'BOTH')
    for doc in docs:
        # 텍스트 정제
        cleaned = clean_text(doc.page_content)
        # 노이즈 행 제거
        lines   = [ln for ln in cleaned.split('\n') if not is_noise_line(ln)]
        cleaned = '\n'.join(lines)
        # search_tags 추가
        doc.page_content = enrich_tricare_text(cleaned, fname)
        # 메타데이터 추가
        doc.metadata['source_file'] = fname
        doc.metadata['location']    = loc
    all_pdf_docs.extend(docs)
    print(f'  ✅ {fname} [{loc}]: {len(docs)}페이지')

print(f'\n📄 총 PDF 페이지 합계: {len(all_pdf_docs)}페이지')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)
print(f'\n--- 샘플 page_content (앞 400자) ---')
print(all_pdf_docs[0].page_content[:400])


---
## Section 5 · PDF 청킹 — 단락 단위 분리

### 개요
- [변경] RecursiveCharacterTextSplitter → **단락(`\n\n`) 단위 직접 분리**로 교체
  - 청크 경계가 문장 중간에서 잘리는 문제 방지
  - 80자 이상 단락만 Document로 생성 (너무 짧은 단락 제거)
- RecursiveCharacterTextSplitter는 CSV 청킹에만 재사용됨 (Section 6)
- 단락 분리 후에도 `source_file`, `location` 메타데이터는 그대로 유지됨


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument

# CSV 청킹용 text_splitter는 그대로 유지 (Section 6에서 재사용)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# PDF는 단락(\n\n) 단위 직접 분리 — 문장 중간 절단 방지
# 80자 미만 단락은 헤더·페이지 번호 등 노이즈일 가능성이 높으므로 제거
MIN_CHUNK_CHARS = 80

pdf_chunks = []
for doc in all_pdf_docs:
    # search_tags 블록은 분리하지 않고 단락 분리 전에 제거 후 각 청크에 다시 붙임
    # (search_tags가 모든 청크에 포함되어야 유사도 향상 효과가 있음)
    full_text = doc.page_content
    if '[search_tags]' in full_text:
        main_text, tags_block = full_text.split('[search_tags]', 1)
        tags_block = '[search_tags]' + tags_block
    else:
        main_text  = full_text
        tags_block = ''

    paragraphs = [p.strip() for p in main_text.split('\n\n') if len(p.strip()) >= MIN_CHUNK_CHARS]
    for para in paragraphs:
        chunk_text = para + ('\n\n' + tags_block if tags_block else '')
        pdf_chunks.append(LCDocument(
            page_content=chunk_text,
            metadata=doc.metadata.copy()
        ))

print(f'분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'(단락 단위 분리, {MIN_CHUNK_CHARS}자 미만 제거)')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:400])
print(f'\nMetadata: {pdf_chunks[0].metadata}')


---
## Section 6 · CSV 로드 — 텍스트 벡터 저장소용 3종

### 개요
- integrated2와 동일한 방식으로 유지함
- `CSVLoader` 대신 직접 변환 방식을 사용함
  - 청킹 후에도 서비스명·항목명이 각 청크에 남아 출처 추적이 가능함

### 3종 파일별 변환 방식
| 파일 | page_content 구성 | metadata 키 |
|---|---|---|
| `mental_health_services.csv` | `서비스: {서비스}\n내용: {내용}` | `source`, `서비스명`, `type` |
| `TricarePlans.csv` | `항목: {Plans}\n플랜명: {플랜명}\n내용: {내용}` | `source`, `plan_name`, `type` |
| `tricare_exclusions.csv` | `제외항목: {name}\n내용: {content}` | `source`, `exclusion_name`, `type`, `url` |


In [ ]:
import pandas as pd
from langchain_core.documents import Document

# 1. mental_health_services.csv
# 서비스명을 page_content에 직접 포함시켜 청킹 후에도 서비스명이 남도록 함
df_mental = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')
mental_docs = []
for _, row in df_mental.iterrows():
    mental_docs.append(Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",
        metadata={
            'source': 'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type': 'mental_health'
        }
    ))

mental_chunks = text_splitter.split_documents(mental_docs)
print(f'📋 mental_health_services: {len(df_mental)}행 → {len(mental_chunks)}청크')


📋 mental_health_services: 8행 → 12청크


In [ ]:
# 2. TricarePlans.csv
# 컬럼 구조: Plans(행 항목) + 각 플랜명(열)
# 플랜별로 항목-내용 쌍을 Document로 변환해 플랜 비교 질문에 검색되도록 함
df_plans = pd.read_csv(CSV_PLANS, encoding='utf-8-sig')
plan_cols = [c for c in df_plans.columns if c != 'Plans']
plans_docs = []
for _, row in df_plans.iterrows():
    topic = row['Plans']
    for plan_name in plan_cols:
        content = str(row[plan_name]).strip()
        if not content or content == 'nan':
            continue
        plans_docs.append(Document(
            page_content=f"항목: {topic}\n플랜명: {plan_name}\n내용: {content}",
            metadata={
                'source': 'TricarePlans.csv',
                'plan_name': plan_name,
                'type': 'plan_info'
            }
        ))

plans_chunks = text_splitter.split_documents(plans_docs)
print(f'📋 TricarePlans: {len(df_plans)}행 × {len(plan_cols)}플랜 → {len(plans_chunks)}청크')


📋 TricarePlans: 10행 × 16플랜 → 164청크


In [ ]:
# 3. tricare_exclusions.csv — 크롤러(tricare_exclusions_crawler) 대체
# 기존에는 크롤러를 실행해 실시간으로 가져왔으나 CSV로 대체함
# url은 metadata에만 저장해 출처 추적에 활용함
df_excl = pd.read_csv(CSV_EXCLUSIONS, encoding='utf-8-sig')
exclusion_docs = []
for _, row in df_excl.iterrows():
    exclusion_docs.append(Document(
        page_content=f"제외항목: {row['name']}\n내용: {row['content']}",
        metadata={
            'source': 'tricare_exclusions.csv',
            'exclusion_name': row['name'],
            'type': 'exclusion',
            'url': row['url']
        }
    ))

exclusion_chunks = text_splitter.split_documents(exclusion_docs)
print(f'📋 tricare_exclusions: {len(df_excl)}행 → {len(exclusion_chunks)}청크')


📋 tricare_exclusions: 55행 → 64청크


---
## Section 7 · 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)

### 개요
- PDF 텍스트 + CSV 3종(정신건강 / 플랜 정보 / 제외 항목)을 하나의 저장소에 통합함
- Retriever가 소스 구분 없이 가장 관련 있는 청크를 한 번에 검색함
- `metadata.source`와 `metadata.type`으로 청크 출처는 항상 추적 가능함

### 통합 청크 구성
| 소스 | 청크 변수 |
|---|---|
| PDF 텍스트 15종 | `pdf_chunks` |
| mental_health_services.csv | `mental_chunks` |
| TricarePlans.csv | `plans_chunks` |
| tricare_exclusions.csv | `exclusion_chunks` |


In [ ]:
from langchain_chroma import Chroma

# PDF + CSV 3종 모두 합산
all_text_chunks = pdf_chunks + mental_chunks + plans_chunks + exclusion_chunks

print(f'📦 청크 합산')
print(f'PDF 텍스트 ({len(RAG_PDF_DOCS)}종):      {len(pdf_chunks)}개')
print(f'mental_health_services: {len(mental_chunks)}개')
print(f'TricarePlans:           {len(plans_chunks)}개')
print(f'tricare_exclusions:     {len(exclusion_chunks)}개')
print(f'합계:                   {len(all_text_chunks)}개')
print('⏳ 벡터 저장소 생성 중...')

# PERSIST_TEXT / COLLECTION_TEXT는 Section 1에서 정의한 상수
vector_store = Chroma.from_documents(
    documents=all_text_chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_TEXT,
    persist_directory=PERSIST_TEXT
)

print(f'✅ 텍스트 벡터 저장소 생성 완료: {vector_store._collection.count()}개 벡터')


📦 청크 합산
PDF 텍스트 (15종):      1172개
mental_health_services: 12개
TricarePlans:           164개
tricare_exclusions:     64개
합계:                   1412개
⏳ 벡터 저장소 생성 중...
✅ 텍스트 벡터 저장소 생성 완료: 4419개 벡터


---
## Section 8 · PDF 표 추출 — pdfplumber + `Health_Plan_Costs.csv`

### 개요
- [변경] 기존 `Costs_Fees.pdf` 단일 대상 → `TABLE_PDF_DOCS` 리스트 순회 방식으로 확장함
  - `Pharmacy_HB.pdf`, `NGR_HB.pdf`, `TFL_HB.pdf`, `Plans_Overview_FS.pdf` 표 추가 수록
- `pdfplumber`는 표의 셀 좌표를 인식해 `행 | 열` 구조를 보존함
- 표는 텍스트 벡터 저장소와 분리해 별도 저장함
  - 비용 질문 시 텍스트 청크가 섞이면 노이즈가 될 수 있음
- `Health_Plan_Costs.csv`는 행 자체가 비용 단위로 구성되어 있어 추가 청킹 불필요함


In [ ]:
import pdfplumber

def extract_tables_from_pdf(pdf_path: str) -> list:
    """
    pdfplumber로 PDF 전체에서 표를 추출함.
    [변경] normalize_cell() 적용 — ✓/✗ → Covered/Not covered 정규화
    [변경] 빈 행 제거 — 모든 셀이 N/A인 행은 노이즈이므로 제외
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있음.
    source_file 메타데이터로 어느 PDF에서 왔는지 팀 통합 시 구분 가능함.
    """
    table_docs = []
    file_name  = os.path.basename(pdf_path)
    loc        = LOCATION_MAP.get(file_name, 'BOTH')
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                lines = []
                for row in table:
                    # normalize_cell 적용 — ✓/✗/빈칸을 텍스트로 변환
                    cleaned_row = [
                        normalize_cell(str(cell).strip().replace('\n', ' ') if cell else '')
                        for cell in row
                    ]
                    # 모든 셀이 N/A인 행은 빈 행이므로 제거
                    if all(c == 'N/A' for c in cleaned_row):
                        continue
                    lines.append(' | '.join(cleaned_row))
                if not lines:
                    continue
                table_docs.append(LCDocument(
                    page_content=f'[TRICARE 비용표 | {file_name} | p.{page_num + 1}]\n' + '\n'.join(lines),
                    metadata={
                        'source':      pdf_path,
                        'source_file': file_name,
                        'page':        page_num + 1,
                        'type':        'cost_table',
                        'table_idx':   table_idx + 1,
                        'location':    loc,
                    }
                ))
    return table_docs

# TABLE_PDF_DOCS 리스트를 순회하며 일괄 표 추출
table_docs_pdf = []
for pdf_path in TABLE_PDF_DOCS:
    docs = extract_tables_from_pdf(pdf_path)
    table_docs_pdf.extend(docs)
    print(f'  📊 {os.path.basename(pdf_path)}: {len(docs)}개 표 추출')

print(f'\n📊 PDF 표 합계: {len(table_docs_pdf)}개')
print('\n--- PDF 표 샘플 ---')
if table_docs_pdf:
    print(table_docs_pdf[0].page_content[:300])


In [ ]:
# Health_Plan_Costs.csv — 비용 전용 저장소용
# 탭(플랜·그룹 정보) + 비용 유형 + 플랜 1/2 수치를 하나의 page_content로 구성
# pdfplumber 표와 동일한 저장소에 넣어 비용 질문의 검색 커버리지를 높임
df_costs = pd.read_csv(CSV_COSTS, encoding='utf-8-sig')
cost_docs = []
for _, row in df_costs.iterrows():
    tab       = str(row['탭']).strip()
    cost_type = str(row['비용 유형']).strip()
    plan1     = str(row['플랜 1']).strip()
    plan2     = str(row['플랜 2']).strip() if str(row['플랜 2']) != 'nan' else ''

    content = f"플랜/그룹: {tab}\n비용 유형: {cost_type}\n플랜 1: {plan1}"
    if plan2:
        content += f"\n플랜 2: {plan2}"

    cost_docs.append(Document(
        page_content=content,
        metadata={
            'source': 'Health_Plan_Costs.csv',
            'cost_type': cost_type,
            'type': 'cost_table'
        }
    ))

# Health_Plan_Costs는 행 자체가 비용 단위로 구성되어 있어 추가 청킹 불필요함
print(f'📊 Health_Plan_Costs: {len(df_costs)}행 → {len(cost_docs)}개 Document')
print('\n--- CSV 비용 샘플 ---')
print(cost_docs[0].page_content)
print(cost_docs[1].page_content)
print(cost_docs[2].page_content)
print(cost_docs[3].page_content)
print(cost_docs[4].page_content)


📊 Health_Plan_Costs: 234행 → 234개 Document

--- CSV 비용 샘플 ---
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Enrollment fee (annual)
플랜 1: $0
플랜 2: $0
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Deductible (annual)
플랜 1: $0
플랜 2: E-1 through E-4: $50 per individual $100 per family E-5 & above: $150 per individual $300 per family
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Catastrophic cap (annual)
플랜 1: $1,000 per family
플랜 2: $1,000 per family
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Clinical preventive services
플랜 1: Network: $0 Non-network: Point-of-service fees
플랜 2: Network and Non-network: $0
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Outpatient visit — primary care
플랜 1: Network:

---
## Section 9 · 표 전용 벡터 저장소 생성 (`chroma_db2`)

### 개요
- PDF 표 + `Health_Plan_Costs.csv`를 하나의 표 전용 저장소에 통합함
- `k=3`: 표 데이터는 정보 밀도가 높아 3개면 충분함
- 비용 질문 시 텍스트 청크가 섞이지 않아 수치 답변 정확도가 높아짐

### 재실행 시 벡터 저장소 불러오는 방법
```python
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)
```


In [ ]:
# PDF 표 + CSV 비용 Document 합산
all_table_docs = table_docs_pdf + cost_docs

print(f'📦 표 데이터 합산')
print(f'   PDF 표 (pdfplumber, {len(TABLE_PDF_DOCS)}종): {len(table_docs_pdf)}개')
print(f'   Health_Plan_Costs.csv:              {len(cost_docs)}개')
print(f'   합계:                               {len(all_table_docs)}개')
print('⏳ 표 전용 벡터 저장소 생성 중...')

# PERSIST_TABLE / COLLECTION_TABLE은 Section 1에서 정의한 상수
table_vector_store = Chroma.from_documents(
    documents=all_table_docs,
    embedding=embedding_model,
    collection_name=COLLECTION_TABLE,
    persist_directory=PERSIST_TABLE
)

print(f'✅ 표 전용 벡터 저장소 생성 완료: {table_vector_store._collection.count()}개 벡터')


📦 표 데이터 합산
   PDF 표 (pdfplumber, 5종): 50개
   Health_Plan_Costs.csv:              234개
   합계:                               284개
⏳ 표 전용 벡터 저장소 생성 중...
✅ 표 전용 벡터 저장소 생성 완료: 860개 벡터


---
## Section 10 · Retriever 설정 — MMR 검색

### 개요
- [변경] `similarity` → **MMR(Maximal Marginal Relevance)** 검색으로 교체
  - 중복 청크 감소, 다양한 문서에서 검색 가능
  - `fetch_k=20`으로 20개 후보 중 다양성 기준으로 k=6 선택
- 텍스트 retriever: k=6, fetch_k=20
- 표 retriever: k=3, fetch_k=10 (표 데이터는 밀도 높아 3개면 충분)
- `normalize_question()`: LLM이 질문에서 intent/region/영어쿼리를 추출해 검색 정확도 향상
  - 비용이 발생하므로 RAG 검색 단계에서만 호출


In [ ]:
from langchain_openai import ChatOpenAI

# normalize_question: LLM이 intent/region/영어쿼리 추출
# 팀원 rag_utils.py의 normalize_question()을 TRICARE 기준으로 수정
# gpt-4o-mini 호출 비용 발생 → RAG 체인 내부에서만 호출
_norm_model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

NORMALIZE_PROMPT = """
You are a TRICARE insurance query analyzer.
Given the user's question, extract:
1. intent: one of [coverage, eligibility, cost, pharmacy, dental, overseas, general]
2. region: one of [CONUS, OCONUS, korea, unknown]
3. english_query: rewrite the question in clear English (for vector search)

Respond ONLY in JSON format:
{{"intent": "...", "region": "...", "english_query": "..."}}

Question: {question}
"""

import json as _json

def normalize_question(question: str) -> dict:
    """질문에서 intent/region/영어쿼리 추출 — 검색 정확도 향상용"""
    try:
        resp = _norm_model.invoke(NORMALIZE_PROMPT.format(question=question))
        return _json.loads(resp.content)
    except Exception:
        # 파싱 실패 시 원문 질문으로 fallback
        return {'intent': 'general', 'region': 'unknown', 'english_query': question}


# MMR 검색 함수 — as_retriever 대신 max_marginal_relevance_search 직접 호출
# normalize_question으로 영어 쿼리 추출 → 검색 정확도 향상
def mmr_retrieve(question: str, store, k: int = 6, fetch_k: int = 20) -> list:
    """MMR 검색: fetch_k개 후보 중 다양성 기준으로 k개 선택"""
    norm   = normalize_question(question)
    query  = norm.get('english_query', question)
    region = norm.get('region', 'unknown')
    docs   = store.max_marginal_relevance_search(query, k=k, fetch_k=fetch_k)
    # OCONUS 질문이면 location 필터 적용 (CONUS 전용 문서 제외)
    if region in ('OCONUS', 'korea'):
        docs = [d for d in docs if d.metadata.get('location', 'BOTH') != 'CONUS'] or docs
    return docs


# 기존 as_retriever도 유지 (make_rag_chain 호환성)
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 6, 'fetch_k': 20}
)
table_retriever = table_vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10}
)

# 동작 확인
test_result = retriever.invoke('TRICARE Prime 본인부담금')
print(f'🔍 MMR retriever 테스트: {len(test_result)}개 검색됨')
for i, d in enumerate(test_result):
    print(f'  [{i+1}] {d.metadata.get("source_file", d.metadata.get("source", "?"))} | loc={d.metadata.get("location","?")} : {d.page_content[:60]}...')

# normalize_question 테스트
test_q = '한국에서 TRICARE Prime 외래 진료 본인부담금이 얼마예요?'
norm_result = normalize_question(test_q)
print(f'\n🔍 normalize_question 테스트:')
print(f'  intent:        {norm_result["intent"]}')
print(f'  region:        {norm_result["region"]}')
print(f'  english_query: {norm_result["english_query"]}')


---
## Section 11 · LLM + RAG 체인 구성

### 개요
- integrated2와 동일하게 유지함 (팀원 통일 사항)
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 출처 인용 형식과 통일함

  **⬇️⬇️⬇️⬇️ 수정함***

  ## Section 11 · LLM + RAG 체인 구성

### 개요
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 출처 인용 형식과 통일함
- [변경] OCONUS 컨텍스트 추가: 주한미군·한국 거주 수혜자 대상 시스템임을 프롬프트에 명시함
  - 한국에서는 트라이케어가 1차 지불자, Medicare 해외 적용 불가 등 핵심 규칙 주입


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 언어 감지 함수 (팀원 통일 방식 그대로 유지)
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': "the same language as the user's question",
}

# format_docs: 출처 정보(source_file, p.페이지)를 청크 앞에 붙임
# search_tags 블록은 LLM 컨텍스트에서 제거 — LLM에게 태그 텍스트가 보이면 노이즈
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source_file', doc.metadata.get('source', 'unknown'))
        page   = doc.metadata.get('page', '')
        label  = f'[{source}{", p." + str(page) if page else ""}]'
        # search_tags 블록 제거 후 LLM에 전달
        content = doc.page_content
        if '[search_tags]' in content:
            content = content.split('[search_tags]')[0].strip()
        result.append(f'{label}\n{content}')
    return '\n\n'.join(result)


def make_rag_chain(question: str, retriever, model):
    """
    [변경] normalize_question으로 영어 쿼리 추출 → mmr_retrieve 사용
    retriever 인자는 호환성 유지를 위해 그대로 받되, 내부적으로 store 직접 접근
    """
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요. '
        '정보 제공만 하고 최종 판단은 사용자 본인과 보험사에게 맡기세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official terms. '
        f'Use term locking format: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )


# mmr_retrieve를 활용한 make_rag_chain_v2 — normalize_question 완전 통합 버전
# Demo에서는 make_rag_chain(기존 호환) 또는 make_rag_chain_v2 중 선택 가능
def make_rag_chain_v2(question: str, store, model):
    """normalize_question + mmr_retrieve 완전 통합 버전"""
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')
    norm   = normalize_question(question)
    docs   = mmr_retrieve(question, store)
    context = format_docs(docs)

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        f'Query intent: {norm.get("intent", "general")} | Region: {norm.get("region", "unknown")}\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- Use term locking: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        f'[참고 문서]\n{context}\n\n'
        f'[질문]\n{question}\n\n'
        '[답변]\n'
    )

    from langchain_core.messages import HumanMessage
    resp = model.invoke([HumanMessage(content=PROMPT_TEMPLATE)])
    return resp.content, docs


print('✅ LLM + RAG 체인 구성 완료 (v2 — normalize_question + MMR 통합)')
print('   make_rag_chain(q, retriever, model)  → 기존 호환 버전')
print('   make_rag_chain_v2(q, store, model)   → normalize_question + MMR 완전 통합 버전')


---
## Section 12 · Demo 1 — 일반 LLM vs RAG 비교

### 개요
- 같은 질문을 일반 LLM과 RAG에 각각 던져 답변 품질을 비교함
- 일반 LLM: 학습 데이터 기반으로 답변 → 세부 조건 누락 가능
- RAG: 실제 문서에서 정확한 내용을 검색 후 답변
- 출력 형식: `QUESTION → ANSWER → RETRIEVED DOCS` 순서로 통일함


In [ ]:
from langchain_core.messages import HumanMessage

question = (
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? '
    '세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.'
)

print('=' * 100)
print('QUESTION:', question)

print('\n❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답')
print('-' * 65)
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print('\n⚠️  주의: 위 답변은 학습 데이터 기반이며 실제 문서와 다를 수 있음')

print('\n✅ [Demo 1 - RAG 답변] — 문서 기반 응답')
print('-' * 65)
rag_chain = make_rag_chain(question, retriever, model)
rag_answer = rag_chain.invoke(question)

print('=== ANSWER ===')
print(rag_answer)

print('\n=== RETRIEVED DOCS ===')
for doc in retriever.invoke(question):
    print(doc.metadata)


QUESTION: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? 세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.

❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답
-----------------------------------------------------------------
TRICARE Prime에서 정신건강 상담(Psychotherapy)은 다양한 형태로 제공되며, 보장 내용은 다음과 같습니다:

### 보장 내용
1. **개인 상담 (Individual Therapy)**: 개인 상담은 개인의 정신적, 정서적 문제를 다루기 위한 세션입니다.
2. **가족 상담 (Family Therapy)**: 가족 구성원 간의 관계 문제를 해결하기 위한 세션입니다.
3. **그룹 상담 (Group Therapy)**: 여러 사람이 함께 참여하여 서로의 경험을 공유하고 지원하는 세션입니다.

### 시간 제한
- **개인 상담**: 일반적으로 세션은 50분에서 60분 정도 진행됩니다. 연간 세션 수는 개인의 필요에 따라 다르지만, TRICARE는 보통 20회까지 보장합니다.
- **가족 상담**: 가족 상담도 비슷한 시간 제한이 있으며, 세션당 50분에서 60분 정도 진행됩니다. 연간 세션 수는 개인 상담과 유사하게 보장됩니다.
- **그룹 상담**: 그룹 상담은 보통 1시간에서 2시간 정도 진행되며, 그룹의 크기와 주제에 따라 다를 수 있습니다.

### 주의사항
- **사전 승인**: TRICARE Prime에서는 특정 치료에 대해 사전 승인이 필요할 수 있습니다. 따라서 상담을 시작하기 전에 담당 의사와 상담하여 필요한 절차를 확인하는 것이 중요합니다.
- **네트워크 제공자 이용**: TRICARE Prime의 혜택을 최대한 활용하기 위해서는 TRICARE 네트워크에 등록된 정신건강 전문가를 이용해야 합니다. 네트워크 외의 제공자를 이용할 경우 추가 비용이 발생할 수 있습니다.


---
## Section 13 · Demo 2 — 텍스트 RAG Q&A

### 개요
- `retriever`(텍스트+CSV 통합)를 사용해 다양한 질문 유형을 테스트함
- 신규 추가 PDF에서 답변이 검색되는지 확인하는 용도로도 활용 가능함
- 출력 형식: `QUESTION → === ANSWER === → === RETRIEVED DOCS ===` 순서로 통일함


In [ ]:
questions = [
    # 기존 질문
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?',
    # 신규 PDF 커버리지 확인용 질문
    'TRICARE Pharmacy Home Delivery를 이용하려면 어떻게 등록하나요?',      # Pharmacy_HB
    '국가방위군이 활성화(Activated)될 때 TRICARE 플랜은 어떻게 변경되나요?', # NGR_HB
    'TRICARE For Life에서 Medicare와 비용은 어떻게 분담되나요?',            # TFL_HB
    'TRICARE ADDP에서 크라운(Crown)은 사전승인이 필요한가요?',              # TRICARE_ADDP_HB
    '임신 중 다른 지역으로 이사할 때 TRICARE 보장은 어떻게 되나요?',         # Maternity_Br
    '65세가 되면 TRICARE Prime에서 어떤 플랜으로 전환되나요?',              # Medicare_Turning_65_Br
    'QLE(Qualifying Life Event)에 해당하는 사례는 어떤 것들이 있나요?',     # QLEs_FS
]

for q in questions:
    print('=' * 100)
    print('QUESTION:', q)

    # 질문마다 언어 감지 후 체인 생성
    rag_chain = make_rag_chain(q, retriever, model)
    answer = rag_chain.invoke(q)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in retriever.invoke(q):
        print(doc.metadata)
    print()


QUESTION: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?
=== ANSWER ===
TRICARE Prime에서는 정신건강 상담(Psychotherapy)이 의학적 또는 심리적으로 필요할 경우 사무실 기반의 외래 정신건강 치료로 보장됩니다. TRICARE는 입원 및 외래 서비스 모두를 포함하며, 개인 세션은 최대 60분까지 보장됩니다. 위기 상황에서의 개인 세션은 최대 120분까지 연장될 수 있습니다.

=== RETRIEVED DOCS ===
{'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
{'서비스명': 'Psychotherapy', 'type': 'mental_health', 'source': 'mental_health_services.csv'}
{'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
{'type': 'mental_health', '서비스명': 'Psychotherapy', 'source': 'mental_health_services.csv'}
{'source': 'TricarePlans.csv', 'type': 'plan_info', 'plan_name': 'TRICARE Prime Demo'}
{'source': 'TricarePlans.csv', 'plan_name': 'TRICARE Prime Demo', 'type': 'plan_info'}

QUESTION: TRICARE Pharmacy Home Delivery를 이용하려면 어떻게 등록하나요?
=== ANSWER ===
TRICARE Pharmacy Home Delivery를 이용하려면 다음 방법 중 하나로 등록할 수 있습니다:

1. **온라인**: www.esrx.com/mhd를 방문하여 등록합니다.
2.

---
## Section 14 · Demo 3 — 표 전용 비용 Q&A

### 개요
- `table_retriever`(표 전용)를 사용해 비용 수치를 정확하게 검색함
- 텍스트 청크가 섞이지 않아 `$26`, `$45` 같은 수치 답변이 더 정확함
- 복수 질문을 리스트로 넣어 한 번에 실행 가능함


In [ ]:
cost_questions = [
    'TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?',
    'TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?',
    'TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?',
    'TRICARE For Life에서 Medicare만 보장하는 서비스 이용 시 본인 부담은 얼마인가요?', # 신규 TFL_HB 표 활용
]

for q in cost_questions:
    print('=' * 100)
    print('QUESTION:', q)

    # 질문마다 언어 감지 후 체인 생성
    table_rag_chain = make_rag_chain(q, table_retriever, model)
    answer = table_rag_chain.invoke(q)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in table_retriever.invoke(q):
        print(doc.metadata)
    print()


QUESTION: TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?
=== ANSWER ===
TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 네트워크 또는 비네트워크 모두 $79입니다.

=== RETRIEVED DOCS ===
{'cost_type': 'Emergency visit', 'type': 'cost_table', 'source': 'Health_Plan_Costs.csv'}
{'type': 'cost_table', 'source': 'Health_Plan_Costs.csv', 'cost_type': 'Emergency visit'}
{'source': 'Health_Plan_Costs.csv', 'cost_type': 'Emergency visit', 'type': 'cost_table'}

QUESTION: TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?
=== ANSWER ===
TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 $52입니다.

=== RETRIEVED DOCS ===
{'source': 'Health_Plan_Costs.csv', 'type': 'cost_table', 'cost_type': 'Outpatient visit — specialty care'}
{'type': 'cost_table', 'cost_type': 'Outpatient visit — specialty care', 'source': 'Health_Plan_Costs.csv'}
{'source': 'Health_Plan_Costs.csv', 'cost_type': 'Outpatient visit — specialty care', 'type': 'cost_table'}

QUESTION: TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?
=== ANSWER ===

---
## Section 15 · 정리 및 다음 단계

### 데이터 처리 전략 요약
| 데이터 | 처리 방식 | 저장소 |
|---|---|---|
| PDF 텍스트 15종 | PyMuPDFLoader → 청킹 | `chroma_db` |
| PDF 표 5종 | pdfplumber → Document | `chroma_db2` |
| mental_health_services.csv | 직접 row 변환 → 청킹 | `chroma_db` |
| TricarePlans.csv | 항목×플랜 쌍으로 변환 → 청킹 | `chroma_db` |
| tricare_exclusions.csv | 직접 row 변환 → 청킹 | `chroma_db` |
| Health_Plan_Costs.csv | 직접 row 변환 (청킹 불필요) | `chroma_db2` |

### Demo 결과 요약
| | 일반 LLM | RAG 텍스트 | RAG 표 |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | PDF 15종 + CSV 3종 | PDF 표 5종 + 비용 CSV |
| 정확도 | 낮음 | 높음 | 매우 높음 (수치) |
| 다국어 답변 | 가능 | ✅ 언어 감지 + 프롬프트 주입 | ✅ 언어 감지 + 프롬프트 주입 |
| 출처 표기 | 불가 | ✅ [파일명, p.페이지] | ✅ [파일명, p.페이지] |

### 향후 개선 방향
1. **메타데이터 필터링** — `type`, `plan_name`, `location` 필터를 Retriever에 추가
2. **앙상블 Retriever** — BM25(키워드) + bge-m3(의미) 혼합 검색
3. **평가 파이프라인** — RAGAS 라이브러리로 답변 품질 정량 측정
4. **통합 벡터 DB 병합** — 팀원 각자 벡터 DB를 `add_documents()`로 통합 컬렉션에 병합
